In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

2025-09-30 16:45:42 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Vinculaciones adquirencia

La tabla para obtener a lo largo del tiempo los comercios con adquirencia es tan pequeña que no es necesario crear un script que primero generé un histórico y luego un script que actualice ese histórico. Por tanto, cada vez que se necesite la evolución del kpi, se creará desde la tabla fuente:

`resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf`

In [10]:
dict_ult_ing_vinculacion = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_vinculacion

2025-10-01 08:42:39 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
/Users/santlond/anaconda3/envs/env_odbc_py39/lib/python3.9/site-packages/helper/helper.py:421: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-10-01 08:42:40 - [WARNING] - Error en la busqueda de ultima ingestion para la tabla resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf, intentando nuevamente
2025-10-01 08:42:40 - [INFO] - Finalizo la busqueda, duracion: 00:01.1, resultado: {'year': 2025, 'month': 9, 'day': 26}


{'year': 2025, 'month': 9, 'day': 26}

In [93]:
# Ingestión información de vinculación
# Se debe esperar hasta el siguiente mes para obtener la vinculación del mes actual
sql = """
SELECT ingestion_year, ingestion_month, ingestion_day, year, month, day, periodo, count(*) as frec
FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
WHERE YEAR BETWEEN 2024 and 2025
  AND MONTH  BETWEEN 1 and 12
  AND DAY BETWEEN 1 and 31
GROUP BY 1,2,3,4,5,6,7
ORDER BY periodo desc, year DESC, month desc, day desc, ingestion_year desc, ingestion_month desc, ingestion_day desc;
"""
helper.obtener_dataframe(sql).head(20)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 42/42 DATAFRAME                                            ejecutando   11:26:21 AM             

2025-10-01 11:26:32 - [INFO] - 27 filas, 8 columnas, 00:10.9 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 42/42 DATAFRAME                                            finalizado   11:26:21 AM     00:11.0 
-------------------------------------------------------------------------------------------------


,ingestion_year,ingestion_month,ingestion_day,year,month,day,periodo,frec
0,2025,9,26,2025,9,26,202509.0,1638
1,2025,9,26,2025,9,26,202508.0,4225
2,2025,8,8,2025,8,8,202507.0,3817
3,2025,7,8,2025,7,8,202506.0,3092
4,2025,6,18,2025,6,18,202505.0,3198
5,2025,5,6,2025,5,6,202504.0,3244
6,2025,4,4,2025,4,4,202503.0,2827
7,2025,3,11,2025,3,11,202502.0,2735
8,2025,2,7,2025,2,7,202501.0,2361
9,2025,1,11,2025,1,11,202412.0,3325


In [91]:
# Número de Vinculaciones
sql = """
WITH vinc AS
  (SELECT periodo,
          count(*) AS num_vinc,
          cast(left(cast(periodo AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(periodo AS STRING), 2) AS int) AS mes
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR BETWEEN 2020 AND 2025
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1
   ORDER BY periodo DESC)
SELECT periodo,
       num_vinc,
       sum(num_vinc) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_cumsum_year_mes
FROM vinc
ORDER BY periodo DESC;
"""
helper.obtener_dataframe(sql).head(20)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 40/40 DATAFRAME                                             falló n.1   11:03:36 AM             
 40/40 DATAFRAME                                             intento 2   11:03:36 AM             

2025-10-01 11:04:29 - [INFO] - 69 filas, 5 columnas, 00:52.5 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 40/40 DATAFRAME                                            finalizado   11:03:36 AM     00:52.7 
-------------------------------------------------------------------------------------------------


,periodo,year,mes,num_vinc,num_vinc_cumsum_year_mes
0,202509.0,2025,9,1638,27137
1,202508.0,2025,8,4225,25499
2,202507.0,2025,7,3817,21274
3,202506.0,2025,6,3092,17457
4,202505.0,2025,5,3198,14365
5,202504.0,2025,4,3244,11167
6,202503.0,2025,3,2827,7923
7,202502.0,2025,2,2735,5096
8,202501.0,2025,1,2361,2361
9,202412.0,2024,12,3325,41666


# Uso adquirencia o activo en adquirencia

Un comercio usa adquirencia en un mes específico o mes de análisis cuando:

- Métrica Normal: tienen al menos 1 trx aporbada en ese mes.

- Métrica 5x: tiene al menos 1 trx aprobada en los últimos seis meses, incluyendo el mes de análisis


## Análisis ingestión compras tabla transaccional adquirencia

In [95]:
# Se deben esperar dos días para que se ingeste la información completa de las transacciones de un día en particular
# Para obtener la información de las compras del día jueves y viernes se debe espear hasta la sgte semana
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          left(cast(f_trx as string), 10) as f_trx,
          count(*) AS num_compras
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
   WHERE YEAR IN (2025)
     AND MONTH BETWEEN 8 AND 9
     AND DAY BETWEEN 1 AND 31
     and tipo_trx = "Purchase"
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
helper.obtener_dataframe(sql)[0:20]

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 44/44 DATAFRAME                                            ejecutando   11:29:37 AM             

2025-10-01 11:29:54 - [INFO] - 1,510 filas, 10 columnas, 00:16.6 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 44/44 DATAFRAME                                            finalizado   11:29:37 AM     00:16.8 
-------------------------------------------------------------------------------------------------


,year,month,day,f_trx,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
0,2025,9,11,2025-10-01,1,1,1,1,1.0000,1.0000
1,2025,9,29,2025-09-29,34,34,34,1,1.0000,1.0000
2,2025,9,29,2025-09-28,1890063,1890063,1890063,1,1.0000,1.0000
3,2025,9,29,2025-09-27,2301178,2301178,2301178,1,1.0000,1.0000
4,2025,9,29,2025-09-26,2001121,2001177,2001177,2,1.0000,1.0000
5,2025,9,26,2025-09-26,56,2001177,56,1,0.0000,0.0000
6,2025,9,29,2025-09-25,51417,1783604,1783604,3,0.0288,1.0000
7,2025,9,26,2025-09-25,1732143,1783604,1732187,2,0.9711,0.9712
8,2025,9,25,2025-09-25,44,1783604,44,1,0.0000,0.0000
9,2025,9,29,2025-09-24,112,1653032,1653032,4,0.0001,1.0000


## Construcción histórico transacciones

In [ ]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2022-09-01' # MODIFICAR.
fecha_final = '2025-09-05' # MODIFICAR.
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='D')

# Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
pri_dia_part = fechas[-1] + relativedelta(days=1)
pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
ult_dia_part = fechas[-1] + relativedelta(days=10)
ult_dia_part = ult_dia_part.date().isoformat()
fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config

,fechas,year,month,day
0,2022-09-01,2022,9,1
1,2022-09-02,2022,9,2
2,2022-09-03,2022,9,3
3,2022-09-04,2022,9,4
4,2022-09-05,2022,9,5
...,...,...,...,...
117,2022-12-27,2022,12,27
118,2022-12-28,2022,12,28
119,2022-12-29,2022,12,29
120,2022-12-30,2022,12,30


In [ ]:
# # Crear tabla que almacenará la información

# sql = """
# CREATE TABLE proceso_bluekai.mdo_adquirencia_trxs  (
#                 cod_unico VARCHAR,
#                 f_trx TIMESTAMP,
#                 num_trxs BIGINT,
#                 mnt_total_trxs DECIMAL(38,2)
#                 )
#             PARTITIONED BY 
#             (
#             YEAR INT,
#             MES INT,
#             DIA INT
#             )
# STORED AS PARQUET
# TBLPROPERTIES ('transactional' = 'false');
# """
# helper.ejecutar_consulta(sql)

# sql_compute = """COMPUTE STATS proceso_bluekai.mdo_adquirencia_trxs;"""

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 53/53      DROP proceso_bluekai.mdo_adquirencia_trxs   finalizado   07:47:50 AM     00:01.2 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 54/54    CREATE proceso_bluekai.mdo_adquirencia_trxs   finalizado   07:47:51 AM     00:00.1 
---------------------------------------------------------------------------------------------


In [37]:
# Iterar para obtener las trxs por cliente
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month), '-', str(row.day))
    print('')
    print('Obteniendo transacciones adquirencia de los comercios')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    SELECT cod_unico,
       f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs,
        """ + str(row.year) + """ AS YEAR,
        """ + str(row.month) + """ AS MES,
        """ + str(row.day) + """ AS DIA
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = """ + str(row.year) + """
     AND MONTH = """ + str(row.month) + """
     AND DAY = """ + str(row.day) + """
     AND tipo_trx = "Purchase"
    GROUP BY 1,
            2;"""
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando transacciones adquirencia de los comercios')
    print('')

    sql = """
    INSERT INTO proceso_bluekai.mdo_adquirencia_trxs PARTITION (YEAR, MES, DIA)
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           YEAR,
           MES,
           DIA
    FROM proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql)
    print('')

##################################################

Exrayendo datos de las particiones:  2022 - 9 - 1

Obteniendo transacciones adquirencia de los comercios

-------------------------------------------------------------------------------------------------
     i       tipo                   nombre                   estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 3981/3981      DROP    proceso.mdo_adquirencia_trxs_temp    falló n.1   11:52:35 AM             
 3981/3981      DROP    proceso.mdo_adquirencia_trxs_temp   finalizado   11:52:35 AM     00:01.7 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
     i       tipo                   nombre                   estado     hora_inicio   duracion   
----------------------------------------------------------

In [8]:
# Verificar cantidad de registros por partición
# La última ingestión usada 2025-05-05 para obtener las transacciones.
sql = """
SELECT YEAR,
       mes,
       dia,
       left(cast(f_trx AS string), 10),
       count(*) AS frec
FROM proceso_bluekai.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND 2025
  AND mes BETWEEN 1 AND 12
  AND dia BETWEEN 1 AND 31
GROUP BY 1,
         2,
         3,
         4
ORDER BY left(cast(f_trx AS string), 10) DESC, YEAR DESC, mes DESC,
                                                          dia DESC;
"""
helper.obtener_dataframe(sql).head(20)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 6/6 DATAFRAME        descargando   04:56:40 PM             

2025-09-30 16:56:44 - [INFO] - 27,679 filas, 5 columnas, 00:03.5 consultando, 00:00.7 descargando, 00:00.0 convirtiendo


 6/6 DATAFRAME         finalizado   04:56:40 PM     00:04.4 
------------------------------------------------------------


,year,mes,dia,expr_1,frec
0,2025,8,26,2025-09-19,1
1,2025,8,26,2025-09-16,1
2,2025,9,5,2025-09-05,4
3,2025,9,5,2025-09-04,410078
4,2025,9,4,2025-09-04,3
5,2025,9,5,2025-09-03,16341
6,2025,9,4,2025-09-03,432313
7,2025,9,3,2025-09-03,4
8,2025,9,5,2025-09-02,94
9,2025,9,4,2025-09-02,15441


## Construcción histórico transacciones por mes

In [114]:
# Crear tabla que almacenará las transacciones menusales por cliente
sql = """
CREATE TABLE proceso_bluekai.mdo_adquirencia_trxs_mes_1  (
                cod_unico VARCHAR,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2)
                )
            PARTITIONED BY 
            (
            periodo_trxs INT
            )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso_bluekai.mdo_adquirencia_trxs_mes_1;"""

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 47/47    CREATE ...so_bluekai.mdo_adquirencia_trxs_mes_1   finalizado   03:35:29 PM     00:00.5 
-------------------------------------------------------------------------------------------------


In [116]:
# Construcción histórico trxs mes
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2022-01-01' # MODIFICAR.
fecha_final = '2025-09-30' # MODIFICAR.
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='MS')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['fechas_fin_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1) - relativedelta(days=1))
df_config['year_fin_mes'] = df_config['fechas_fin_mes'].dt.year
df_config['month_fin_mes'] = df_config['fechas_fin_mes'].dt.month
df_config['day_fin_mes'] = df_config['fechas_fin_mes'].dt.day
df_config

,fechas,year,month,day,fechas_fin_mes,year_fin_mes,month_fin_mes,day_fin_mes
0,2022-01-01,2022,1,1,2022-01-31,2022,1,31
1,2022-02-01,2022,2,1,2022-02-28,2022,2,28
2,2022-03-01,2022,3,1,2022-03-31,2022,3,31
3,2022-04-01,2022,4,1,2022-04-30,2022,4,30
4,2022-05-01,2022,5,1,2022-05-31,2022,5,31
5,2022-06-01,2022,6,1,2022-06-30,2022,6,30
6,2022-07-01,2022,7,1,2022-07-31,2022,7,31
7,2022-08-01,2022,8,1,2022-08-31,2022,8,31
8,2022-09-01,2022,9,1,2022-09-30,2022,9,30
9,2022-10-01,2022,10,1,2022-10-31,2022,10,31


In [120]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month))
    print('')
    print('Obteniendo transacciones adquirencia de los comercios')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_mes_1_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_mes_1_temp STORED AS PARQUET AS
    SELECT cod_unico,
           cast(replace(left(cast(f_trx AS string), 7), '-', '') AS INT) AS periodo_trxs,
           count(*) AS num_trxs,
           sum(mnt_total_trxs) AS mnt_total_trxs
    FROM proceso_bluekai.mdo_adquirencia_trxs
    WHERE YEAR = """ + str(row.year) + """
     AND MES = """ + str(row.month) + """
     AND DIA BETWEEN """ + str(row.day) + """ AND """ + str(row.day_fin_mes) + """
    GROUP BY 1,
             2;"""
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_mes_1_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando transacciones adquirencia de los comercios')
    print('')

    sql = """
    INSERT INTO proceso_bluekai.mdo_adquirencia_trxs_mes_1 PARTITION (periodo_trxs)
    SELECT cod_unico,
           num_trxs,
           mnt_total_trxs,
           periodo_trxs
    FROM proceso.mdo_adquirencia_trxs_mes_1_temp;"""
    helper.ejecutar_consulta(sql)
    print('')

##################################################

Exrayendo datos de las particiones:  2022 - 1

Obteniendo transacciones adquirencia de los comercios

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 56/56      DROP  proceso.mdo_adquirencia_trxs_mes_1_temp   finalizado   03:52:26 PM     00:00.5 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 57/57    CREATE  proceso.mdo_adquirencia_trxs_mes_1_temp   fi

## Construcción histórico métrica 5x

In [132]:
# Tabla que almacenará comercios con trxs por periodo
sql = """
CREATE TABLE proceso_bluekai.mdo_adquirencia_vinculaciones_con_trxs_hist  (
                codigo_unico DOUBLE,
                periodo DOUBLE
                )
            PARTITIONED BY 
            (
            YEAR INT,
            MES INT
            )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso_bluekai.mdo_adquirencia_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 237/237    CREATE ...quirencia_vinculaciones_con_trxs_hist   finalizado   04:41:49 PM     00:00.5 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 238/238   COMPUTE ...quirencia_vinculaciones_con_trxs_hist   finalizado   04:41:50 PM     00:00.8 
---------------------------------------------------------------------------------------------------


In [78]:
df_config.ult_6_meses_fin[0].replace(day=1)

Timestamp('2022-08-01 00:00:00')

In [124]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2023-01-01' # MODIFICAR.
fecha_final = '2025-08-31' # MODIFICAR.
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)
df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_sgte_mes,year_sgte_mes,ult_6_meses_fin,ult_6_meses_inicio,year_ult_6_meses,month_ult_6_meses,periodo_ult_6_meses
0,2023-01-31,2023,1,31,202301,202301,202302,2023,2022-08-31,2022-08-01,2022,8,202208
1,2023-02-28,2023,2,28,202302,202301,202303,2023,2022-09-28,2022-09-01,2022,9,202209
2,2023-03-31,2023,3,31,202303,202301,202304,2023,2022-10-31,2022-10-01,2022,10,202210
3,2023-04-30,2023,4,30,202304,202301,202305,2023,2022-11-30,2022-11-01,2022,11,202211
4,2023-05-31,2023,5,31,202305,202301,202306,2023,2022-12-31,2022-12-01,2022,12,202212
5,2023-06-30,2023,6,30,202306,202301,202307,2023,2023-01-30,2023-01-01,2023,1,202301
6,2023-07-31,2023,7,31,202307,202301,202308,2023,2023-02-28,2023-02-01,2023,2,202302
7,2023-08-31,2023,8,31,202308,202301,202309,2023,2023-03-31,2023-03-01,2023,3,202303
8,2023-09-30,2023,9,30,202309,202301,202310,2023,2023-04-30,2023-04-01,2023,4,202304
9,2023-10-31,2023,10,31,202310,202301,202311,2023,2023-05-31,2023-05-01,2023,5,202305


In [134]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
    WHERE YEAR BETWEEN """ + str(row.year) + """ AND """ + str(row.year_sgte_mes) + """
    AND MONTH BETWEEN 1 AND 12
    AND DAY BETWEEN 1 AND 31
    AND periodo BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """ ;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones seis meses hacia atrás')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select distinct cod_unico
    FROM proceso_bluekai.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_ult_6_meses) + """ AND """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_bluekai.mdo_adquirencia_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_bluekai.mdo_adquirencia_vinculaciones_con_trxs_hist PARTITION (YEAR,
                                                                                       MES)
    SELECT a.codigo_unico,
           a.periodo,
           """ + str(row.year) + """ AS YEAR,
           """ + str(row.month) + """ AS MES
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2023 - 1

Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 241/241      DROP ...so.mdo_adquirencia_vinculaciones_temp   finalizado   04:44:33 PM     00:00.4 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 242/242   

In [139]:
# Obtener uso por mes
# Número de Vinculaciones
sql = """
SELECT periodo,
       count(*) AS num_vinc_usando_5x
FROM proceso_bluekai.mdo_adquirencia_vinculaciones_con_trxs_hist
WHERE YEAR BETWEEN 2020 AND 2025
  AND MES BETWEEN 1 AND 12
  AND periodo IS NOT NULL
GROUP BY 1
ORDER BY periodo DESC;
"""
helper.obtener_dataframe(sql).head(50)


---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 469/469 DATAFRAME                                           descargando   05:03:44 PM             

2025-10-01 17:03:50 - [INFO] - 32 filas, 5 columnas, 00:05.4 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 469/469 DATAFRAME                                            finalizado   05:03:44 PM     00:05.9 
---------------------------------------------------------------------------------------------------


,periodo,year,mes,num_vinc_usando_5x,num_vinc_usando_5x_cumsum_year_mes
0,202508.0,2025,8,17701,72067
1,202507.0,2025,7,14880,54366
2,202506.0,2025,6,12224,39486
3,202505.0,2025,5,9981,27262
4,202504.0,2025,4,7578,17281
5,202503.0,2025,3,5211,9703
6,202502.0,2025,2,3192,4492
7,202501.0,2025,1,1300,1300
8,202412.0,2024,12,28017,191827
9,202411.0,2024,11,25704,163810


# Resultados

In [141]:
sql = """
WITH vinc AS
  (SELECT periodo,
          count(*) AS num_vinc,
          cast(left(cast(periodo AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(periodo AS STRING), 2) AS int) AS mes
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR BETWEEN 2020 AND 2025
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1),
     vinc_acum AS
  (SELECT periodo,
          num_vinc,
          sum(num_vinc) OVER (PARTITION BY YEAR
                              ORDER BY YEAR,
                                       mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_cumsum_year_mes
   FROM vinc),
     vinc_uso AS
  (SELECT periodo,
          count(*) AS num_vinc_usando_5x
   FROM proceso_bluekai.mdo_adquirencia_vinculaciones_con_trxs_hist
   WHERE YEAR BETWEEN 2020 AND 2025
     AND MES BETWEEN 1 AND 12
     AND periodo IS NOT NULL
   GROUP BY 1),
     outcome1 AS
  (SELECT a.periodo,
          a.num_vinc,
          a.num_vinc_cumsum_year_mes,
          b.num_vinc_usando_5x
   FROM vinc_acum AS a
   LEFT JOIN vinc_uso AS b ON cast(a.periodo AS int) = cast(b.periodo AS int)
   WHERE a.periodo <= 202508)
SELECT a.periodo,
       a.num_vinc,
       a.num_vinc_cumsum_year_mes,
       a.num_vinc_usando_5x,
       round(a.num_vinc_usando_5x/a.num_vinc_cumsum_year_mes, 4) AS prop_uso
FROM outcome1 AS a
ORDER BY a.periodo DESC;
"""
helper.obtener_dataframe(sql).to_excel('main_data/evolucion_vinculaciones_y_metrica5x.xlsx', index=False)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 471/471 DATAFRAME                                             falló n.1   09:51:58 AM             
 471/471 DATAFRAME                                           descargando   09:51:58 AM             

2025-10-03 09:52:37 - [INFO] - 68 filas, 5 columnas, 00:37.8 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 471/471 DATAFRAME                                            finalizado   09:51:58 AM     00:38.3 
---------------------------------------------------------------------------------------------------
